# Umsatz-Prognose Projekt

Ziel: Vorhersage von Verkaufsmengen basierend auf verschiedenen Features


## 1. Imports und Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('Alle Imports erfolgreich!')

## 2. Daten generieren

In [ ]:
np.random.seed(42)

# Generiere realistische Verkaufsdaten
n_samples = 500

data = {
    'Monat': np.tile(range(1, 13), n_samples // 12 + 1)[:n_samples],
    'Werbebudget': np.random.uniform(1000, 10000, n_samples),
    'Lagerbestand': np.random.uniform(100, 1000, n_samples),
    'Mitarbeiter': np.random.randint(5, 50, n_samples),
    'Kundenanzahl': np.random.randint(100, 1000, n_samples),
}

# Erzeuge Umsatz mit realistischem Zusammenhang
data['Umsatz'] = (
    0.5 * data['Werbebudget'] +
    2 * data['Lagerbestand'] +
    100 * data['Mitarbeiter'] +
    5 * data['Kundenanzahl'] +
    200 * np.sin(np.array(data['Monat']) * 2 * np.pi / 12) +  # Saisonalität
    np.random.normal(0, 500, n_samples)  # Rauschen
)

df = pd.DataFrame(data)

print(f'Datensatz mit {len(df)} Zeilen erstellt')
print(f'\nShape: {df.shape}')
print(f'\nErste 5 Zeilen:')
df.head()

## 3. Explorative Datenanalyse (EDA)

In [ ]:
print('Datentypen:')
print(df.dtypes)
print(f'\nFehlende Werte: {df.isnull().sum().sum()}')
print(f'\nStatistische Übersicht:')
df.describe()

In [ ]:
# Visualisierung der Verteilungen
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Datenverteilungen', fontsize=16, fontweight='bold')

columns = df.columns
for idx, col in enumerate(columns):
    ax = axes[idx // 3, idx % 3]
    ax.hist(df[col], bins=30, alpha=0.7, color='steelblue', edgecolor='black')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('Wert')
    ax.set_ylabel('Häufigkeit')

plt.tight_layout()
plt.show()

In [ ]:
# Korrelationsmatrix
correlation = df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, cbar_kws={'label': 'Korrelation'})
plt.title('Korrelationsmatrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Korrelation mit Umsatz:')
correlation['Umsatz'].sort_values(ascending=False)

## 4. Feature Engineering & Vorbereitung

In [ ]:
# Features und Target trennen
X = df.drop('Umsatz', axis=1)
y = df['Umsatz']

print(f'Features (X): {X.shape}')
print(f'Target (y): {y.shape}')

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'\nTrainingsmenge: {X_train.shape[0]} Samples')
print(f'Testmenge: {X_test.shape[0]} Samples')

In [ ]:
# Normalisierung
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Daten normalisiert')

## 5. Modelltraining

In [ ]:
# Lineare Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_pred_train = lr_model.predict(X_train_scaled)
lr_pred_test = lr_model.predict(X_test_scaled)

print('Linear Regression trainiert')
print(f'Train R²: {r2_score(y_train, lr_pred_train):.4f}')
print(f'Test R²: {r2_score(y_test, lr_pred_test):.4f}')

In [ ]:
# Random Forest
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred_train = rf_model.predict(X_train)
rf_pred_test = rf_model.predict(X_test)

print('Random Forest trainiert')
print(f'Train R²: {r2_score(y_train, rf_pred_train):.4f}')
print(f'Test R²: {r2_score(y_test, rf_pred_test):.4f}')

## 6. Modell-Evaluierung

In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print(f'{model_name}:')
    print(f'  MSE:  {mse:,.2f}')
    print(f'  RMSE: {rmse:,.2f}')
    print(f'  MAE:  {mae:,.2f}')
    print(f'  R²:   {r2:.4f}')
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

print('=== TRAININGS-METRIKEN ===')
evaluate_model(y_train, lr_pred_train, 'Linear Regression (Train)')
print()
evaluate_model(y_train, rf_pred_train, 'Random Forest (Train)')

print('\n=== TEST-METRIKEN ===')
lr_test_metrics = evaluate_model(y_test, lr_pred_test, 'Linear Regression (Test)')
print()
rf_test_metrics = evaluate_model(y_test, rf_pred_test, 'Random Forest (Test)')

In [ ]:
# Vorhersage vs. Tatsächliche Werte
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear Regression
axes[0].scatter(y_test, lr_pred_test, alpha=0.6, s=50, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Tatsächlicher Umsatz', fontsize=12)
axes[0].set_ylabel('Vorhergesagter Umsatz', fontsize=12)
axes[0].set_title(f'Linear Regression (R²={lr_test_metrics["R2"]:.4f})', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Random Forest
axes[1].scatter(y_test, rf_pred_test, alpha=0.6, s=50, color='green')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Tatsächlicher Umsatz', fontsize=12)
axes[1].set_ylabel('Vorhergesagter Umsatz', fontsize=12)
axes[1].set_title(f'Random Forest (R²={rf_test_metrics["R2"]:.4f})', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
# Feature Importance aus Random Forest
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
bars = plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue')
plt.xlabel('Wichtigkeit', fontsize=12, fontweight='bold')
plt.title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
for i, bar in enumerate(bars):
    width = bar.get_width()
    plt.text(width, bar.get_y() + bar.get_height()/2, f'{width:.3f}', 
             ha='left', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(feature_importance)

## 8. Residuen-Analyse

In [ ]:
# Residuen (Vorhersagefehler)
lr_residuals = y_test - lr_pred_test
rf_residuals = y_test - rf_pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear Regression Residuen
axes[0].scatter(lr_pred_test, lr_residuals, alpha=0.6, color='steelblue', s=50)
axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0].set_xlabel('Vorhergesagter Umsatz', fontsize=12)
axes[0].set_ylabel('Residuen', fontsize=12)
axes[0].set_title('Linear Regression - Residuenplot', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Random Forest Residuen
axes[1].scatter(rf_pred_test, rf_residuals, alpha=0.6, color='green', s=50)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Vorhergesagter Umsatz', fontsize=12)
axes[1].set_ylabel('Residuen', fontsize=12)
axes[1].set_title('Random Forest - Residuenplot', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Linear Regression - Residuen Mittelwert: {lr_residuals.mean():.2f}')
print(f'Linear Regression - Residuen Standardabweichung: {lr_residuals.std():.2f}')
print(f'\nRandom Forest - Residuen Mittelwert: {rf_residuals.mean():.2f}')
print(f'Random Forest - Residuen Standardabweichung: {rf_residuals.std():.2f}')

## 9. Zusammenfassung & Empfehlung

In [ ]:
print('='*60)
print('PROJEKT-ZUSAMMENFASSUNG: UMSATZ-PROGNOSE')
print('='*60)

print('\n📊 DATENSATZ:')
print(f'  • Samples: {len(df)}')
print(f'  • Features: {len(X.columns)} (Werbebudget, Lagerbestand, Mitarbeiter, Kundenanzahl, Monat)')
print(f'  • Target: Umsatz')

print('\n🤖 MODELLE:')
print(f'  • Linear Regression - Test R²: {lr_test_metrics["R2"]:.4f}')
print(f'  • Random Forest - Test R²: {rf_test_metrics["R2"]:.4f}')

print('\n🏆 EMPFEHLUNG:')
if rf_test_metrics['R2'] > lr_test_metrics['R2']:
    print(f'  → Random Forest ist das bessere Modell!')
    print(f'  → R² Unterschied: {(rf_test_metrics["R2"] - lr_test_metrics["R2"]):.4f}')
    print(f'  → Durchschnittlicher Fehler (RMSE): {rf_test_metrics["RMSE"]:.2f} €')
else:
    print(f'  → Linear Regression ist das bessere Modell!')
    
print('\n💡 INSIGHTS:')
print(f'  • Top Feature: {feature_importance.iloc[0, 0]} (Importance: {feature_importance.iloc[0, 1]:.4f})')
print(f'  • Modell erklärt {rf_test_metrics["R2"]*100:.1f}% der Varianz im Umsatz')
print('\n' + '='*60)